# NB1 — Plutchik Labeling: GoEmotions Part 1 (rows 0–21999)
**Person 1 runs this notebook.** Labels the first half of GoEmotions with soft Plutchik scores using Qwen2.5-3B-Instruct (4-bit, no API key needed).

- GPU: T4 x1 (Settings → Accelerator)
- Expected runtime: ~2 hours
- Output: `/kaggle/working/plutchik_labeled_p1.csv`

> After completion: go to Output tab → Download or share as a Kaggle dataset named `plutchik-labels-p1`


## 0. Install

In [ ]:
%%capture
!pip install -U requests -q
!pip install datasets==2.19.0 -q
!pip install transformers accelerate bitsandbytes sentencepiece -q
print('done')

## 1. Config

In [ ]:
import os, json, re, torch
import pandas as pd
import numpy as np
from datasets import load_dataset

PART_START   = 0
PART_END     = 22000      # exclusive — first half
SAVE_PATH    = '/kaggle/working/plutchik_labeled_p1.csv'
CKPT_PATH    = '/kaggle/working/ckpt_p1.csv'  # intermediate checkpoint
MODEL_ID     = 'Qwen/Qwen2.5-3B-Instruct'     # fully open, no HF token needed
BATCH_SIZE   = 16     # T4 has headroom with 4-bit (~2GB model, 16GB VRAM)
MAX_NEW_TOK  = 120    # needs 120 tokens to complete all 9 JSON keys
EMOTIONS     = ['joy','trust','fear','surprise','sadness','disgust','anger','anticipation']
print('Config OK  |  GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Load Model (4-bit NF4 on T4 — ~2 GB VRAM)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit NF4 — requires T4/A100 (compute capability >= 7.5)
# Do NOT use P100 — it is Pascal (6.0) and does not support 4-bit bitsandbytes
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token

print('Loading model (4-bit NF4)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
    low_cpu_mem_usage=True,   # load layer-by-layer to avoid RAM spike
    trust_remote_code=True,
)
model.eval()
print(f'Model loaded | device: {next(model.parameters()).device}')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 3. Prompt + Parser

In [ ]:
def build_prompt(sentence: str) -> str:
    system = (
        'Output ONLY a JSON object with keys: '
        'joy, trust, fear, surprise, sadness, disgust, anger, anticipation, confidence. '
        'All values are floats 0.0-1.0. No explanation.'
    )
    return (
        f'<|im_start|>system\n{system}<|im_end|>\n'
        f'<|im_start|>user\n{sentence.strip()}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )


def parse_output(raw: str) -> dict | None:
    """Robust parser — handles single quotes, wrong case, extra text around JSON."""
    raw = raw.strip()

    # Find the first {...} block in the output
    m = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
    if not m:
        return None
    try:
        blob = m.group()
        # Fix single quotes -> double quotes
        blob = blob.replace("'", '"')
        # Fix unquoted keys: word: -> "word":
        blob = re.sub(r'(?<!["\w])(\w+)\s*:', r'"\1":', blob)
        # Fix doubled quotes from above fix: '""word""' -> '"word"'
        blob = re.sub(r'""(\w+)""', r'"\1"', blob)

        obj = json.loads(blob)

        # Case-insensitive key matching
        obj_lower = {k.lower().strip(): v for k, v in obj.items()}

        required = set(EMOTIONS + ['confidence'])
        if not required.issubset(obj_lower.keys()):
            return None

        return {k: max(0.0, min(1.0, float(obj_lower[k]))) for k in required}
    except Exception:
        return None


print('Prompt + parser ready')

## 4. Batched Inference

In [ ]:
from tqdm.auto import tqdm


def label_batch(sentences: list) -> list:
    prompts = [build_prompt(s) for s in sentences]
    inputs = tokenizer(
        prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=192,
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOK,
            do_sample=False,              # greedy — faster + more consistent JSON
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    results = []
    prompt_len = inputs['input_ids'].shape[1]
    for seq in out:
        raw = tokenizer.decode(seq[prompt_len:], skip_special_tokens=True)
        results.append(parse_output(raw))
    return results


def label_dataset(sentences: list, batch_size: int = BATCH_SIZE) -> pd.DataFrame:
    rows = []
    n_fail = 0
    for i in tqdm(range(0, len(sentences), batch_size), desc='Labeling'):
        batch = sentences[i:i+batch_size]
        results = label_batch(batch)
        for sent, res in zip(batch, results):
            if res is not None:
                rows.append({'text': sent, **res})
            else:
                n_fail += 1
        # Checkpoint every 500 successful rows
        if len(rows) > 0 and len(rows) % 500 < batch_size:
            pd.DataFrame(rows).to_csv(CKPT_PATH, index=False)
    print(f'Parse failures: {n_fail}/{len(sentences)} ({100*n_fail/max(len(sentences),1):.1f}%)')
    return pd.DataFrame(rows)


print('Inference functions ready')

## 5. Load GoEmotions (rows 0–21999)

In [ ]:
print('Loading GoEmotions...')
go = load_dataset('go_emotions', 'simplified', split='train')
all_texts = [row['text'] for row in go]

sentences = all_texts[PART_START:PART_END]
print(f'Sentences to label: {len(sentences):,} (index {PART_START}–{PART_END-1})')

## 6. Quick Test (3 sentences)

In [ ]:
test_sents = [
    'She was absolutely thrilled to see him but terrified of what came next.',
    "I can't believe they did that. It's disgusting and makes me furious.",
    'When she finally...',
]
print('Quick test:\n')
for s in test_sents:
    r = label_batch([s])[0]
    if r:
        emo_str = '  '.join(f'{k}={v:.2f}' for k,v in r.items() if k != 'confidence')
        print(f'Input  : {s}')
        print(f'Output : [{emo_str}]  conf={r["confidence"]:.2f}')
    else:
        print(f'Input  : {s}  -> PARSE FAILED')
    print()

## 7. Full Labeling Run
> This will take ~1.5–2 hours for 22K sentences. Progress saves to `ckpt_p1.csv` every 500 rows.

In [ ]:
# Resume from checkpoint if it exists
if os.path.exists(CKPT_PATH):
    ckpt_df = pd.read_csv(CKPT_PATH)
    done_texts = set(ckpt_df['text'].tolist())
    sentences_todo = [s for s in sentences if s not in done_texts]
    print(f'Resuming from checkpoint: {len(ckpt_df):,} already done, {len(sentences_todo):,} remaining')
else:
    sentences_todo = sentences
    ckpt_df = pd.DataFrame()
    print(f'Starting fresh: {len(sentences_todo):,} sentences')

new_df = label_dataset(sentences_todo, batch_size=BATCH_SIZE)
labeled_df = pd.concat([ckpt_df, new_df], ignore_index=True)
print(f'Total labeled: {len(labeled_df):,}')

## 8. Quality Analysis

In [ ]:
clean = labeled_df.dropna(subset=EMOTIONS)
print(f'Parse success : {len(clean):,} / {len(labeled_df):,} ({100*len(clean)/max(len(labeled_df),1):.1f}%)')
print(f'Mean confidence: {clean["confidence"].mean():.3f}')
print(f'Low conf (<0.2) : {(clean["confidence"] < 0.2).mean()*100:.1f}%')
print('\nMean ± Std per emotion:')
for e in EMOTIONS:
    print(f'  {e:<15}: {clean[e].mean():.3f} ± {clean[e].std():.3f}')

## 9. Filter & Save

In [ ]:
# Keep sentences with successful parse and confidence >= 0.2
final = clean[clean['confidence'] >= 0.2].copy().reset_index(drop=True)
final.to_csv(SAVE_PATH, index=False)
print(f'Final dataset : {len(final):,} sentences')
print(f'Saved to      : {SAVE_PATH}')
print()
print('DONE. Share this notebook output as a Kaggle dataset named: plutchik-labels-p1')
print('Person 3 will add it as input to NB3.')
final.head(5)